In [2]:
import os
import numpy as np
import pandas as pd
import tifffile
from scipy.interpolate import CubicSpline

In [ ]:
def normalize_spectra(spectra):
    """Min-max normalize spectra: (n_samples, n_channels) or (n_channels,)."""
    if spectra.ndim == 1:
        min_v = np.min(spectra)
        max_v = np.max(spectra)
        return (spectra - min_v) / (max_v - min_v + 1e-10)
    else:
        min_v = np.min(spectra, axis=1, keepdims=True)
        max_v = np.max(spectra, axis=1, keepdims=True)
        return (spectra - min_v) / (max_v - min_v + 1e-10)


def load_background_spectrum(csv_path, num_samples=60, wn_start=2700.0, wn_end=3100.0):
    """
    Load background spectrum CSV (e.g. water or tissue background).
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Background CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)

    if df.shape[1] >= 2:
        col0 = df.iloc[:, 0].dropna().values
        col1 = df.iloc[:, 1].dropna().values
        try:
            wn_vals = col0.astype(float)
            int_vals = col1.astype(float)
            s_idx = np.argsort(wn_vals)
            wn_vals = wn_vals[s_idx]
            int_vals = int_vals[s_idx]
            target_wn = np.linspace(wn_start, wn_end, num_samples)
            cs = CubicSpline(wn_vals, int_vals, extrapolate=True)
            bg_interp = cs(target_wn)
            return normalize_spectra(bg_interp)
        except (ValueError, TypeError):
            pass

    vals = list(df.iloc[:, 0].dropna().values)

    bg_vals = np.array(vals, dtype=np.float32)
    # Flip orientation to align with HSI frequency acquisition
    bg_vals = np.flip(bg_vals)

    if len(bg_vals) == num_samples:
        bg_interp = bg_vals
    elif len(bg_vals) >= 2:
        x_old = np.linspace(0.0, 1.0, len(bg_vals))
        x_new = np.linspace(0.0, 1.0, num_samples)
        cs = CubicSpline(x_old, bg_vals, extrapolate=True)
        bg_interp = cs(x_new)
    else:
        bg_interp = np.full(num_samples, bg_vals[0] if len(bg_vals) > 0 else 1.0)

    return normalize_spectra(bg_interp)

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# 1. Directory and files
roi_dir = r"E:\261024 16-1 Kidney\TopROIs"
hsi_path = os.path.join(roi_dir, "2-781.3-806.5nm-8us.tif")
bg_path = os.path.join(roi_dir, "2-781.3nm-250mW-zoom2.0.tif")

# Target channels (excluding 781.3 and 806.5 as requested)
target_wls = [784.4, 786.5, 791.3, 794.6, 797.2]
window_radius = 2  # radius of 2: 5-channel window [ch - 2 : ch + 3]

# 2. Load hyperspectral image & establish wavelength calibration
hsi = tifffile.imread(hsi_path).astype(np.float32)  # shape: (60, 1024, 1024)
C, H, W = hsi.shape
hsi_wls = np.linspace(781.3, 806.5, C)

# 3. Load single channel images and calculate HSI MIP with radius=2
singles = {}
hsi_mips = {}
ch_indices = {}

for wl in target_wls:
    fname = f"2-{wl:.1f}nm-250mW-zoom2.0.tif"
    singles[wl] = tifffile.imread(os.path.join(roi_dir, fname)).astype(np.float32)
    
    # Locate closest channel in hyperspectral stack
    ch_idx = int(np.argmin(np.abs(hsi_wls - wl)))
    ch_indices[wl] = ch_idx
    ch_start = max(0, ch_idx - window_radius)
    ch_end = min(C, ch_idx + window_radius + 1)
    hsi_mips[wl] = np.max(hsi[ch_start:ch_end], axis=0)

# Baseline images (781.3 nm)
bg_single = tifffile.imread(bg_path).astype(np.float32)
bg_hsi = hsi[0]

print(f"Loaded HSI: shape={hsi.shape}, spectral range={hsi_wls[0]:.1f} - {hsi_wls[-1]:.1f} nm")
for wl in target_wls:
    idx = ch_indices[wl]
    print(f"Target {wl:.1f} nm -> Nearest HSI ch {idx} ({hsi_wls[idx]:.2f} nm), window [{max(0, idx-2)}:{min(C, idx+3)}]")



In [ ]:
# 4. Compare all possible ratio pairs
pairs = list(itertools.combinations(target_wls, 2))
records = []

for num_wl, den_wl in pairs:
    s_num, s_den = singles[num_wl], singles[den_wl]
    h_num, h_den = hsi_mips[num_wl], hsi_mips[den_wl]
    
    # Foreground mask (avoid division by zero or extreme noise floor)
    mask_raw = (s_den > 50) & (h_den > 50)
    r_single = s_num[mask_raw] / (s_den[mask_raw] + 1e-6)
    r_hsi = h_num[mask_raw] / (h_den[mask_raw] + 1e-6)
    
    pr_raw, _ = pearsonr(r_single, r_hsi)
    sr_raw, _ = spearmanr(r_single, r_hsi)
    
    # Baseline subtracted (using 781.3 nm)
    s_num_sub = np.clip(s_num - bg_single, 0, None)
    s_den_sub = np.clip(s_den - bg_single, 0, None)
    h_num_sub = np.clip(h_num - bg_hsi, 0, None)
    h_den_sub = np.clip(h_den - bg_hsi, 0, None)
    
    mask_sub = (s_den_sub > 20) & (h_den_sub > 20)
    if np.sum(mask_sub) > 100:
        r_single_sub = s_num_sub[mask_sub] / (s_den_sub[mask_sub] + 1e-6)
        r_hsi_sub = h_num_sub[mask_sub] / (h_den_sub[mask_sub] + 1e-6)
        pr_sub, _ = pearsonr(r_single_sub, r_hsi_sub)
        sr_sub, _ = spearmanr(r_single_sub, r_hsi_sub)
    else:
        pr_sub, sr_sub = np.nan, np.nan
        
    records.append({
        "Pair (Num / Den)": f"{num_wl} / {den_wl}",
        "Raw Pearson r": round(pr_raw, 4),
        "Raw Spearman rho": round(sr_raw, 4),
        "BG-Sub Pearson r": round(pr_sub, 4) if not np.isnan(pr_sub) else np.nan,
        "BG-Sub Spearman rho": round(sr_sub, 4) if not np.isnan(sr_sub) else np.nan,
    })

df_comparison = pd.DataFrame(records)
df_comparison.sort_values(by="Raw Pearson r", ascending=False, inplace=True)
display(df_comparison)



In [ ]:
# 5. Visual comparison: Ratio maps, 2D correlation, and channel difference maps
fig, axes = plt.subplots(2, 3, figsize=(15, 9), dpi=100)

# Display top-performing pair: 784.4 nm / 794.6 nm
num_wl, den_wl = 784.4, 794.6
r_s = singles[num_wl] / (singles[den_wl] + 1e-6)
r_h = hsi_mips[num_wl] / (hsi_mips[den_wl] + 1e-6)

vmin, vmax = np.percentile(r_s, 2), np.percentile(r_s, 98)

axes[0, 0].imshow(r_s, vmin=vmin, vmax=vmax, cmap="magma")
axes[0, 0].set_title(f"Single Channel Ratio: {num_wl} / {den_wl}")
axes[0, 0].axis("off")

axes[0, 1].imshow(r_h, vmin=vmin, vmax=vmax, cmap="magma")
axes[0, 1].set_title(f"HSI MIP (r=2) Ratio: {num_wl} / {den_wl}")
axes[0, 1].axis("off")

mask = (singles[den_wl] > 50) & (hsi_mips[den_wl] > 50)
axes[0, 2].hexbin(r_s[mask], r_h[mask], gridsize=50, cmap="viridis", bins="log", mincnt=1)
axes[0, 2].set_xlabel(f"Single Channel Ratio ({num_wl}/{den_wl})")
axes[0, 2].set_ylabel(f"HSI Window MIP Ratio ({num_wl}/{den_wl})")
axes[0, 2].set_title("Hexbin Correlation Density")

# Individual channel comparison (e.g. 794.6 nm)
axes[1, 0].imshow(singles[794.6], cmap="gray")
axes[1, 0].set_title("Single Channel (794.6 nm)")
axes[1, 0].axis("off")

axes[1, 1].imshow(hsi_mips[794.6], cmap="gray")
axes[1, 1].set_title("HSI MIP radius 2 (794.6 nm)")
axes[1, 1].axis("off")

# Mean-normalized difference
norm_s = singles[794.6] / np.mean(singles[794.6])
norm_h = hsi_mips[794.6] / np.mean(hsi_mips[794.6])
im = axes[1, 2].imshow(norm_s - norm_h, cmap="bwr", vmin=-1.0, vmax=1.0)
axes[1, 2].set_title("Normalized Difference (Single - HSI MIP)")
axes[1, 2].axis("off")
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()



In [ ]:
# 6. Normalized Ratios: (797.2 / (791.3 + 797.2)) and (786.5 / (786.5 + 794.6))
# Evaluated with and without background subtraction (781.3 nm)

formulas = [
    {
        "name": "797.2 / (791.3 + 797.2)",
        "num": 797.2,
        "den_components": [791.3, 797.2]
    },
    {
        "name": "786.5 / (786.5 + 794.6)",
        "num": 786.5,
        "den_components": [786.5, 794.6]
    }
]

norm_records = []
fig, axes = plt.subplots(2, 4, figsize=(18, 9), dpi=100)

for row_idx, form in enumerate(formulas):
    c_num = form["num"]
    c1, c2 = form["den_components"]
    
    # Raw signals
    s_n, s1, s2 = singles[c_num], singles[c1], singles[c2]
    h_n, h1, h2 = hsi_mips[c_num], hsi_mips[c1], hsi_mips[c2]
    
    # --- Without Background Subtraction ---
    s_ratio_raw = s_n / (s1 + s2 + 1e-6)
    h_ratio_raw = h_n / (h1 + h2 + 1e-6)
    
    mask_raw = ((s1 + s2) > 100) & ((h1 + h2) > 100)
    pr_raw, _ = pearsonr(s_ratio_raw[mask_raw], h_ratio_raw[mask_raw])
    sr_raw, _ = spearmanr(s_ratio_raw[mask_raw], h_ratio_raw[mask_raw])
    
    # --- With Background Subtraction (781.3 nm) ---
    s_n_sub = np.clip(s_n - bg_single, 0, None)
    s1_sub = np.clip(s1 - bg_single, 0, None)
    s2_sub = np.clip(s2 - bg_single, 0, None)
    
    h_n_sub = np.clip(h_n - bg_hsi, 0, None)
    h1_sub = np.clip(h1 - bg_hsi, 0, None)
    h2_sub = np.clip(h2 - bg_hsi, 0, None)
    
    s_ratio_sub = s_n_sub / (s1_sub + s2_sub + 1e-6)
    h_ratio_sub = h_n_sub / (h1_sub + h2_sub + 1e-6)
    
    mask_sub = ((s1_sub + s2_sub) > 50) & ((h1_sub + h2_sub) > 50)
    if np.sum(mask_sub) > 100:
        pr_sub, _ = pearsonr(s_ratio_sub[mask_sub], h_ratio_sub[mask_sub])
        sr_sub, _ = spearmanr(s_ratio_sub[mask_sub], h_ratio_sub[mask_sub])
    else:
        pr_sub, sr_sub = np.nan, np.nan
        
    norm_records.append({
        "Formula": form["name"],
        "Condition": "Raw (No BG Sub)",
        "Pearson r": round(pr_raw, 4),
        "Spearman rho": round(sr_raw, 4),
        "Single Mean +/- Std": f"{s_ratio_raw[mask_raw].mean():.3f} +/- {s_ratio_raw[mask_raw].std():.3f}",
        "HSI MIP Mean +/- Std": f"{h_ratio_raw[mask_raw].mean():.3f} +/- {h_ratio_raw[mask_raw].std():.3f}",
    })
    norm_records.append({
        "Formula": form["name"],
        "Condition": "BG Subtracted (781.3 nm)",
        "Pearson r": round(pr_sub, 4) if not np.isnan(pr_sub) else np.nan,
        "Spearman rho": round(sr_sub, 4) if not np.isnan(sr_sub) else np.nan,
        "Single Mean +/- Std": f"{s_ratio_sub[mask_sub].mean():.3f} +/- {s_ratio_sub[mask_sub].std():.3f}" if np.sum(mask_sub) > 100 else "N/A",
        "HSI MIP Mean +/- Std": f"{h_ratio_sub[mask_sub].mean():.3f} +/- {h_ratio_sub[mask_sub].std():.3f}" if np.sum(mask_sub) > 100 else "N/A",
    })
    
    # Plotting: Single Raw vs HSI Raw, and BG-Sub versions
    v0, v1 = np.percentile(s_ratio_raw, 2), np.percentile(s_ratio_raw, 98)
    axes[row_idx, 0].imshow(s_ratio_raw, vmin=v0, vmax=v1, cmap="magma")
    axes[row_idx, 0].set_title(f"Single Raw: {form['name']}", fontsize=10)
    axes[row_idx, 0].axis("off")
    
    axes[row_idx, 1].imshow(h_ratio_raw, vmin=v0, vmax=v1, cmap="magma")
    axes[row_idx, 1].set_title(f"HSI MIP Raw: {form['name']}", fontsize=10)
    axes[row_idx, 1].axis("off")
    
    axes[row_idx, 2].hexbin(s_ratio_raw[mask_raw], h_ratio_raw[mask_raw], gridsize=40, cmap="viridis", bins="log", mincnt=1)
    axes[row_idx, 2].set_xlabel("Single Channel Ratio")
    axes[row_idx, 2].set_ylabel("HSI MIP Ratio")
    axes[row_idx, 2].set_title(f"Raw Correlation: r = {pr_raw:.3f}", fontsize=10)
    
    if np.sum(mask_sub) > 100:
        axes[row_idx, 3].hexbin(s_ratio_sub[mask_sub], h_ratio_sub[mask_sub], gridsize=40, cmap="plasma", bins="log", mincnt=1)
        axes[row_idx, 3].set_title(f"BG-Sub Correlation: r = {pr_sub:.3f}", fontsize=10)
    else:
        axes[row_idx, 3].text(0.5, 0.5, "Insufficient Signal", ha="center")
    axes[row_idx, 3].set_xlabel("Single (BG Sub)")
    axes[row_idx, 3].set_ylabel("HSI MIP (BG Sub)")

plt.tight_layout()
plt.show()

df_norm = pd.DataFrame(norm_records)
display(df_norm)

